# Expertiment No. 9

**Name:** Siddhi Shashikant Gandhi

**Roll No.** 33

**Batch**: T2



In [ ]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense, Dropout, BatchNormalization
from keras.regularizers import l2
import time
import pickle
import os

import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [ ]:
with open('/content/SampleDataset (1).txt', 'r', encoding='utf-8') as f:
    data = f.read().lower().split('\n')
take = int(len(data) * 0.8)
data_small = data[:take]

print("Using lines:", take)
print("Example lines:", data_small[:3])
print("Total lines in file:", len(data))
print("First 5 lines:\n", data[:5])


Using lines: 2468
Example lines: ['the sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees. people were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank. children were playing games, and laughter filled the air.', '', 'as the day turned into evening, the temperature started to drop, and the sky transformed into a canvas of vibrant colors. families gathered for picnics, and the smell of barbecues wafted through the air. it was a perfect day for a picnic by the lake.']
Total lines in file: 3086
First 5 lines:
 ['the sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees. people were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank. children were playing games, and laughter filled the air.', '', 'as the day turned into evening, the temperature started to dro

In [ ]:
# Open the text file in read mode with UTF-8 encoding
with open('/content/SampleDataset (1).txt', 'r', encoding='latin-1') as file:
    # Read the contents of the file and assign it to the variable 'data'
    data = file.read()

In [ ]:
def separate_punc(doc_text):
    """
    Separate punctuation from the input text and convert tokens to lowercase.

    Args:
    doc_text (str): Input text document.

    Returns:
    list: List of tokens (words) without punctuation, converted to lowercase.
    """
    return [token.lower() for token in doc_text.split(" ") if token not in '\n\n \n\n\n!"-#$%&()--.*+,-/:;<=>?@[\\]^_`{|}~\t\n ']

In [ ]:
# Clean the text data by removing punctuation and converting tokens to lowercase
data = separate_punc(data)

# Join the cleaned tokens back into a single string
cleaned_data = " ".join(data)

In [ ]:
# Initialize a tokenizer object
tokenizer = Tokenizer(num_words=None, char_level=False)

# Fit the tokenizer on the cleaned text data
tokenizer.fit_on_texts([cleaned_data])

# Retrieve the word indices from the tokenizer
tokenizer.word_index

{'the': 1,
 'and': 2,
 'a': 3,
 'of': 4,
 'to': 5,
 'i': 6,
 'you': 7,
 'in': 8,
 'is': 9,
 'monica': 10,
 'it': 11,
 'with': 12,
 'ross': 13,
 'that': 14,
 'rachel': 15,
 'for': 16,
 'chandler': 17,
 'this': 18,
 'on': 19,
 'joey': 20,
 'was': 21,
 'oh': 22,
 'phoebe': 23,
 'are': 24,
 'all': 25,
 'as': 26,
 'what': 27,
 'be': 28,
 'like': 29,
 'no': 30,
 "it's": 31,
 "i'm": 32,
 'her': 33,
 'they': 34,
 'just': 35,
 'from': 36,
 'okay': 37,
 'not': 38,
 'so': 39,
 'my': 40,
 'have': 41,
 'me': 42,
 'where': 43,
 'know': 44,
 'she': 45,
 'we': 46,
 'out': 47,
 'well': 48,
 'their': 49,
 'can': 50,
 'at': 51,
 'he': 52,
 'yeah': 53,
 'your': 54,
 'about': 55,
 'but': 56,
 'its': 57,
 'up': 58,
 "don't": 59,
 'text': 60,
 'scene': 61,
 'by': 62,
 'do': 63,
 'an': 64,
 'or': 65,
 'were': 66,
 'there': 67,
 'if': 68,
 'uh': 69,
 'look': 70,
 'life': 71,
 'through': 72,
 'into': 73,
 'him': 74,
 'his': 75,
 "you're": 76,
 'hey': 77,
 'how': 78,
 'right': 79,
 'think': 80,
 'time': 81,
 'no

In [ ]:
# Calculate the length of the word index generated by the tokenizer
word_index_length = len(tokenizer.word_index)

# Print the length of the word index with a descriptive message
print(f"The length of the word index is: {word_index_length}")

The length of the word index is: 4993


In [ ]:
# Initialize an empty list to store input sequences
input_sequences = []

# Iterate over each sentence in the cleaned data
for sentence in cleaned_data.split('\n'):
    # Tokenize the sentence
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

    # Iterate over the tokenized sentence to create input sequences
    for i in range(1, len(tokenized_sentence)):
        # Append the input sequence to the list
        input_sequences.append(tokenized_sentence[:i+1])

# Print the first few input sequences for demonstration
print(input_sequences[:20])

[[1, 155], [1, 155, 21], [1, 155, 21, 2368], [1, 155, 21, 2368, 1549], [1, 155, 21, 2368, 1549, 8], [1, 155, 21, 2368, 1549, 8, 1], [1, 155, 21, 2368, 1549, 8, 1, 422], [1, 155, 21, 2368, 1549, 8, 1, 422, 692], [1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215], [1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215, 2], [1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215, 2, 3], [1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215, 2, 3, 2369], [1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215, 2, 3, 2369, 1550], [1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215, 2, 3, 2369, 1550, 2370], [1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215, 2, 3, 2369, 1550, 2370, 1], [1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215, 2, 3, 2369, 1550, 2370, 1, 423], [1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215, 2, 3, 2369, 1550, 2370, 1, 423, 4], [1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215, 2, 3, 2369, 1550, 2370, 1, 423, 4, 1], [1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215, 2, 3, 2369, 1550, 2370, 1, 423, 4, 1, 1142], [1, 155, 21, 2368, 1549, 8, 1, 42

In [ ]:
# Calculate the maximum sequence length among all input sequences
max_len = max([len(x) for x in input_sequences])

# Print the maximum sequence length
print(max_len)

325


In [ ]:
# Pad the input sequences to ensure uniform length
padded_input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

# Extract features (X) and labels (y)
X = padded_input_sequences[:, :-1]
y = padded_input_sequences[:, -1]

# Print the shapes of X and y
print(X.shape)
print(y.shape)

(26383, 324)
(26383,)


In [ ]:
# Perform one-hot encoding on the labels (y)
y = to_categorical(y, num_classes=len(tokenizer.word_index) + 1)

# Print the shape of y after one-hot encoding
print(y.shape)

(26383, 4994)


In [ ]:
# Define the LSTM language model architecture
model = Sequential()
model.add(Embedding(4994, 100, input_length=324))
model.add(LSTM(150))
model.add(Dense(4994, activation="softmax"))

# Compile the model
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

# Print the model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train the LSTM language model
model.fit(X, y, epochs=150)

Epoch 1/150
825/825 ━━━━━━━━━━━━━━━━━━━━ 23s 22ms/step - accuracy: 0.0399 - loss: 0.0750
Epoch 2/150
825/825 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.0499 - loss: 0.0016
Epoch 3/150
825/825 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.0487 - loss: 0.0016
Epoch 4/150
825/825 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.0471 - loss: 0.0016
Epoch 5/150
825/825 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.0516 - loss: 0.0016
Epoch 6/150
825/825 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.0478 - loss: 0.0016
Epoch 7/150
825/825 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.0461 - loss: 0.0016
Epoch 8/150
825/825 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.0490 - loss: 0.0016
Epoch 9/150
825/825 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.0490 - loss: 0.0016
Epoch 10/150
825/825 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.0512 - loss: 0.0016
Epoch 11/150
825/825 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.0475 - loss: 0.0016
Epoch 12/150
825/82

In [ ]:
text = "Please let me know"

for i in range(15):
    # Tokenize the input text
    token_text = tokenizer.texts_to_sequences([text])[0]
    # Pad the tokenized text
    padded_token_text = pad_sequences([token_text], maxlen=324, padding='pre')
    # Predict the next word index
    pos = np.argmax(model.predict(padded_token_text))

    # Retrieve the word corresponding to the predicted index
    for word, index in tokenizer.word_index.items():
        if index == pos:
            # Append the predicted word to the input text
            text = text + " " + word
            print(text)
            # Simulate typing effect with a delay of 1 second
            time.sleep(1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step
Please let me know if
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Please let me know if you
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Please let me know if you could
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
Please let me know if you could get
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
Please let me know if you could get to
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
Please let me know if you could get to meet
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Please let me know if you could get to meet the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Please let me know if you could get to meet the guy
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Please let me know if you could get to meet the guy there's
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Please let me know if you could get to meet the guy there's gotta
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Please let me know if you could get to meet the guy there's gotta be
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Please let me know if you could get to meet th

In [ ]:
# Save the trained model to an HDF5 file
model.save('LSTM_model.h5')

# Save the tokenizer to a file using pickle
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Flask Code

In [ ]:
from flask import Flask, render_template, request
import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

app = Flask(__name__)

# Load model
model = load_model("LSTM_model.h5")

# Load tokenizer
with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

context_length = 3   # same as training
total_words = len(tokenizer.word_index) + 1


def generate_text(seed_text, next_words=20):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=context_length, truncating='pre')

        predicted = np.argmax(model.predict(token_list, verbose=0))

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text


@app.route("/", methods=["GET", "POST"])
def home():
    generated_text = ""

    if request.method == "POST":
        seed_text = request.form["seed"]
        generated_text = generate_text(seed_text)

    return render_template("index.html", output=generated_text)


if __name__ == "__main__":
    app.run(debug=True)

# HTML Code

In [ ]:
<!DOCTYPE html>
<html>
<head>
    <title>LSTM Text Generator</title>

    <style>
        body {
            font-family: Arial, sans-serif;
            background: linear-gradient(135deg, #667eea, #764ba2);
            display: flex;
            justify-content: center;
            align-items: center;
            height: 100vh;
            margin: 0;
        }

        .container {
            background: white;
            padding: 40px;
            width: 500px;
            border-radius: 15px;
            box-shadow: 0 15px 30px rgba(0,0,0,0.2);
            text-align: center;
        }

        h1 {
            margin-bottom: 20px;
            color: #333;
        }

        input[type="text"] {
            width: 100%;
            padding: 12px;
            border-radius: 8px;
            border: 1px solid #ccc;
            margin-bottom: 20px;
            font-size: 14px;
        }

        button {
            padding: 12px 25px;
            background: #667eea;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-size: 15px;
            transition: 0.3s ease;
        }

        button:hover {
            background: #5a67d8;
            transform: scale(1.05);
        }

        .output {
            margin-top: 25px;
            padding: 15px;
            background: #f4f4f4;
            border-radius: 10px;
            font-size: 15px;
            min-height: 60px;
        }
    </style>
</head>

<body>

<div class="container">
    <h1>LSTM Text Generator</h1>

    <form method="POST">
        <input type="text" name="seed" placeholder="Enter starting words..." required>
        <button type="submit">Generate Text</button>
    </form>

    <div class="output">
        {{ output }}
    </div>
</div>

</body>
</html>